# 05 — Build yearly Valhalla graphs

Build one graph for each historical GIP PBF, write a per-year manifest, and smoke-test Graz routes with auto and pedestrian costing.

Import and tag the pinned local Valhalla image before running. Existing valid manifests are skipped; invalid or missing graphs are rebuilt only when `OVERWRITE = True`. Graphs are built in a year-specific staging directory and renamed atomically after the route smoke test.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib
import json
import os
import subprocess
import time

import pandas as pd
import requests


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and OGD/.")

# Local configuration: set WSL_DISTRO and the project data locations only if the air-gapped PC differs.
PROJECT_DIR = discover_project_dir()
GIP_DIR = PROJECT_DIR / "OGD" / "GIP"
STATUS_DIR = PROJECT_DIR / "ANAL" / "data" / "routing" / "status"
STATUS_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2026))
PBF_BY_YEAR = {year: GIP_DIR / f"{year}.osm.pbf" for year in YEARS}
GRAPH_STATUS_PATH = STATUS_DIR / "graph_build_status.csv"

# Air-gapped setup: import the pinned image before running this notebook, for example:
#   docker load -i valhalla-scripted-3.8.3.tar
#   docker tag <loaded-image-id> valhalla-scripted:3.8.3
VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:3.8.3"
VALHALLA_IMAGE_ID = "sha256:24ef7955899dececb94e26c6dfb89d64fabfae875f980432694b0261eb6c251b"
VALHALLA_URL = "http://localhost:8002"
# Keep Docker's CPU quota and Valhalla's Mjolnir/server concurrency aligned.
# Four threads is the validated conservative setting for the multimodal GIP graphs.
VALHALLA_CPUS = 4
WSL_EXE = r"C:\Windows\System32\wsl.exe"

# Set this to the distro name shown by `wsl -l -v` in PowerShell.
# Common values are "Ubuntu" or "Ubuntu-24.04". Use None only if your default WSL distro is already correct.
WSL_DISTRO = "Ubuntu"

WSL_PROJECT_ROOT = "$HOME/gruendungsanalyse"
WSL_GRAPH_ROOT = f"{WSL_PROJECT_ROOT}/data/routing/valhalla_graphs"

# Safety switch. Set to True only when you really want to start Docker builds.
RUN_BUILD = True

# Start with one smoke-test year. Change to YEARS after the 2025 build validates.
BUILD_YEARS = YEARS

# Existing valid manifests are skipped unless this is True.
OVERWRITE = True

# Austria graphs can take a while to build. Increase if your machine is slow.
MAX_WAIT_MINUTES = 180

# Graz city test route, lon/lat WGS84.
TEST_ROUTE = [
    {"lon": 15.4395, "lat": 47.0707},
    {"lon": 15.4630, "lat": 47.0580},
]
TEST_COSTINGS = ("auto", "pedestrian")

In [ ]:
def run_local(command: list[str], check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(command, check=check, text=True, capture_output=capture_output)


def wsl_base_command() -> list[str]:
    if WSL_DISTRO:
        return [WSL_EXE, "-d", WSL_DISTRO, "--"]
    return [WSL_EXE, "--"]


def run_wsl(command: str, check: bool = True, capture_output: bool = True) -> subprocess.CompletedProcess:
    return run_local([*wsl_base_command(), "bash", "-lc", command], check=check, capture_output=capture_output)


def require_wsl_distribution() -> None:
    distro_list = run_local([WSL_EXE, "-l", "-q"], check=False)
    combined_output = f"{distro_list.stdout}\n{distro_list.stderr}".lower()
    no_distribution = "no installed distributions" in combined_output or "has no installed distributions" in combined_output
    if distro_list.returncode != 0 and no_distribution:
        raise RuntimeError(
            "WSL is installed, but no Linux distribution is installed yet. "
            "Install Ubuntu first with `wsl --install -d Ubuntu`, restart if Windows asks, "
            "open Ubuntu once to create your Linux user, then rerun this notebook."
        )
    available_distros = [line.strip().replace("\x00", "") for line in distro_list.stdout.splitlines() if line.strip().replace("\x00", "")]
    if WSL_DISTRO and available_distros and WSL_DISTRO not in available_distros:
        raise RuntimeError(
            f"Configured WSL_DISTRO={WSL_DISTRO!r}, but PowerShell reports these WSL distros: {available_distros}. "
            "Set WSL_DISTRO in the config cell to the exact name from `wsl -l -v`."
        )

    test = run_wsl("printf ok", check=False)
    if test.returncode != 0:
        raise RuntimeError(
            "Could not start the configured WSL distribution. Run `wsl -l -v` in PowerShell "
            "and set WSL_DISTRO in this notebook to the exact distro name.\n\n"
            f"stdout:\n{test.stdout}\n\nstderr:\n{test.stderr}"
        )


def quote_bash(value: str) -> str:
    return "'" + value.replace("'", "'\\''") + "'"


def quote_wsl_path(value: str) -> str:
    if value.startswith("$HOME/"):
        return "$HOME/" + quote_bash(value.removeprefix("$HOME/"))
    return quote_bash(value)


def windows_to_wsl_path(path: Path) -> str:
    result = run_local([*wsl_base_command(), "wslpath", "-a", str(path)], check=False)
    if result.returncode == 0 and result.stdout.strip():
        return result.stdout.strip()

    resolved = path.resolve()
    drive = resolved.drive.rstrip(":").lower()
    if drive:
        relative = resolved.relative_to(resolved.anchor).as_posix()
        return f"/mnt/{drive}/{relative}"
    raise RuntimeError(f"Could not convert Windows path to WSL path: {path}\n{result.stderr}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_status() -> pd.DataFrame:
    if GRAPH_STATUS_PATH.exists():
        return pd.read_csv(GRAPH_STATUS_PATH)
    return pd.DataFrame(columns=["year", "status", "started_at", "finished_at", "duration_minutes", "graph_path_wsl", "manifest_path_wsl", "error_message"])


def write_status_row(row: dict) -> None:
    row.setdefault("valhalla_image", VALHALLA_IMAGE)
    row.setdefault("valhalla_image_id", VALHALLA_IMAGE_ID)
    if row.get("year") in PBF_BY_YEAR:
        row.setdefault("gip_snapshot_sha256", sha256_file(PBF_BY_YEAR[int(row["year"])]))
    status = read_status()
    status = status[status["year"] != row["year"]]
    status = pd.concat([status, pd.DataFrame([row])], ignore_index=True)
    status = status.sort_values("year")
    status.to_csv(GRAPH_STATUS_PATH, index=False)


def wsl_graph_dir(year: int, staging: bool = False) -> str:
    suffix = ".building" if staging else ""
    return f"{WSL_GRAPH_ROOT}/{year}{suffix}"


def wsl_manifest_path(year: int, staging: bool = False) -> str:
    return f"{wsl_graph_dir(year, staging=staging)}/build_manifest.json"


def container_name(year: int) -> str:
    return f"co2-valhalla-{year}"

## Preflight Checks

Run this cell first. It confirms that WSL and Docker are reachable and creates the base WSL directory layout.

In [ ]:
missing_pbf = [path for path in PBF_BY_YEAR.values() if not path.exists()]
if missing_pbf:
    raise FileNotFoundError(missing_pbf[0])

require_wsl_distribution()

whoami = run_wsl("whoami").stdout.strip()
docker_version = run_wsl("docker --version").stdout.strip()
image_probe = run_wsl(f"docker image inspect {quote_bash(VALHALLA_IMAGE)} --format '{{{{.Id}}}}'", check=False)
if image_probe.returncode != 0:
    raise RuntimeError(f"Pinned Valhalla image {VALHALLA_IMAGE!r} is not loaded locally. Import the transferred tarball and tag it as {VALHALLA_IMAGE}; this notebook never pulls images.")
loaded_image_id = image_probe.stdout.strip()
if loaded_image_id and loaded_image_id != VALHALLA_IMAGE_ID:
    raise RuntimeError(f"Wrong Valhalla image ID: expected {VALHALLA_IMAGE_ID}, found {loaded_image_id}.")
run_wsl(f"mkdir -p {quote_wsl_path(WSL_GRAPH_ROOT)} {quote_wsl_path(WSL_PROJECT_ROOT + '/logs/graph_builder')} {quote_wsl_path(WSL_PROJECT_ROOT + '/status')}")

print(f"WSL user: {whoami}")
print(docker_version)
print(f"WSL project root: {WSL_PROJECT_ROOT}")
print(f"Graph root: {WSL_GRAPH_ROOT}")

In [ ]:
def graph_manifest_exists(year: int) -> bool:
    test_command = f"test -f {quote_wsl_path(wsl_manifest_path(year))}"
    return run_wsl(test_command, check=False).returncode == 0


def graph_manifest_matches_pbf(year: int) -> bool:
    if not graph_manifest_exists(year):
        return False
    manifest_result = run_wsl(f"cat {quote_wsl_path(wsl_manifest_path(year))}", check=False)
    if manifest_result.returncode != 0:
        return False
    try:
        manifest = json.loads(manifest_result.stdout)
    except json.JSONDecodeError:
        return False
    return manifest.get("gip_snapshot_sha256") == sha256_file(PBF_BY_YEAR[year])


def remove_graph_dir(graph_dir: str) -> None:
    # Valhalla runs as root and creates root-owned tile directories. Clean the bind mount
    # from a short-lived root container, then remove the now-empty directory as the WSL user.
    run_wsl(f"mkdir -p {quote_wsl_path(graph_dir)}")
    cleanup_command = (
        "docker run --rm --entrypoint /bin/sh "
        f"-v {quote_wsl_path(graph_dir)}:/cleanup "
        f"{quote_bash(VALHALLA_IMAGE)} -c 'rm -rf -- /cleanup/* /cleanup/.[!.]* /cleanup/..?*'"
    )
    run_wsl(cleanup_command)
    run_wsl(f"rmdir -- {quote_wsl_path(graph_dir)}")


def prepare_graph_dir(year: int) -> str:
    graph_dir = wsl_graph_dir(year, staging=True)
    # A previous interrupted run may still be writing into this bind-mounted directory.
    run_wsl(f"docker rm -f {quote_bash(container_name(year))} >/dev/null 2>&1 || true", check=False)
    # Only the year-specific staging directory may be removed.
    remove_graph_dir(graph_dir)
    run_wsl(f"mkdir -p {quote_wsl_path(graph_dir)}")
    return graph_dir


def copy_pbf_to_wsl(year: int, graph_dir: str) -> str:
    source_windows = PBF_BY_YEAR[year]
    source_wsl = windows_to_wsl_path(source_windows)
    target_wsl = f"{graph_dir}/{source_windows.name}"
    command = " && ".join([
        f"mkdir -p {quote_wsl_path(graph_dir)}",
        f"test -f {quote_wsl_path(target_wsl)} || cp {quote_bash(source_wsl)} {quote_wsl_path(target_wsl)}",
    ])
    run_wsl(command)
    return target_wsl


def start_valhalla_container(year: int, graph_dir: str) -> None:
    name = container_name(year)
    command = " && ".join([
        f"docker rm -f {quote_bash(name)} >/dev/null 2>&1 || true",
        "docker run -d "
        f"--name {quote_bash(name)} "
        f"--cpus {VALHALLA_CPUS} "
        "-p 8002:8002 "
        f"-e server_threads={VALHALLA_CPUS} "
        "-e build_admins=False "
        "-e build_time_zones=False "
        "-e build_tar=True "
        "-e serve_tiles=True "
        f"-v {quote_wsl_path(graph_dir)}:/custom_files "
        f"{quote_bash(VALHALLA_IMAGE)}",
    ])
    run_wsl(command)


def stop_valhalla_container(year: int) -> None:
    run_wsl(f"docker rm -f {quote_bash(container_name(year))} >/dev/null 2>&1 || true", check=False)


def valhalla_test_routes() -> dict[str, dict]:
    summaries = {}
    for costing in TEST_COSTINGS:
        payload = {"locations": TEST_ROUTE, "costing": costing, "directions_options": {"units": "kilometers"}}
        response = requests.post(f"{VALHALLA_URL}/route", json=payload, timeout=30)
        response.raise_for_status()
        summary = response.json()["trip"]["summary"]
        if summary.get("length", 0) <= 0 or summary.get("time", 0) <= 0:
            raise ValueError(f"Invalid Valhalla {costing} route summary: {summary}")
        summaries[costing] = summary
    return summaries


def wait_until_valhalla_ready(year: int, max_wait_minutes: int = MAX_WAIT_MINUTES) -> dict:
    deadline = time.time() + max_wait_minutes * 60
    last_error = None
    while time.time() < deadline:
        try:
            return valhalla_test_routes()
        except Exception as error:
            last_error = error
            state_result = run_wsl(
                f"docker inspect --format '{{{{.State.Status}}}} {{{{.State.ExitCode}}}}' {quote_bash(container_name(year))}",
                check=False,
            )
            state_parts = state_result.stdout.strip().split()
            if state_parts and state_parts[0] in {"exited", "dead"}:
                logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
                exit_code = state_parts[1] if len(state_parts) > 1 else "unknown"
                raise RuntimeError(
                    f"Valhalla container exited while building {year} (exit code {exit_code})."
                    f"\n\nContainer logs:\n{logs}"
                ) from error
            time.sleep(30)
    logs = run_wsl(f"docker logs --tail 80 {quote_bash(container_name(year))}", check=False).stdout
    raise TimeoutError(f"Valhalla did not become ready for {year}. Last error: {last_error}\n\nContainer logs:\n{logs}")


def write_wsl_manifest(year: int, graph_dir: str, pbf_wsl_path: str, route_summaries: dict[str, dict], started_at: str, finished_at: str) -> str:
    pbf_path = PBF_BY_YEAR[year]
    manifest = {
        "year": year,
        "gip_snapshot_filename": pbf_path.name,
        "gip_snapshot_windows_path": str(pbf_path),
        "gip_snapshot_wsl_path": pbf_wsl_path,
        "gip_snapshot_sha256": sha256_file(pbf_path),
        "graph_path_wsl": wsl_graph_dir(year),
        "valhalla_image": VALHALLA_IMAGE,
        "valhalla_image_id": VALHALLA_IMAGE_ID,
        "profiles": list(TEST_COSTINGS),
        "study_area": "Austria PBF with Styria analysis focus",
        "graph_build_started_at": started_at,
        "graph_build_finished_at": finished_at,
        "test_route_summaries": route_summaries,
        "status": "done",
    }
    manifest_json = json.dumps(manifest, indent=2)
    command = f"cat > {quote_wsl_path(graph_dir + '/build_manifest.json')} <<'EOF'\n{manifest_json}\nEOF"
    run_wsl(command)
    return wsl_manifest_path(year)

## Build Graphs

Keep `RUN_BUILD = False` for a dry run. Set `RUN_BUILD = True` after the preflight cell works. Start with `BUILD_YEARS = [2025]`; after that validates, switch to `BUILD_YEARS = YEARS`.

In [ ]:
for year in BUILD_YEARS:
    started_at = datetime.now().isoformat(timespec="seconds")
    graph_dir = wsl_graph_dir(year)
    staging_dir = wsl_graph_dir(year, staging=True)
    manifest_path = wsl_manifest_path(year)

    if graph_manifest_matches_pbf(year) and not OVERWRITE:
        print(f"Skip {year}: manifest exists at {manifest_path}")
        write_status_row({
            "year": year,
            "status": "done",
            "started_at": started_at,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
            "duration_minutes": 0,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": "",
            "build_action": "skipped_existing_valid",
        })
        continue

    if not OVERWRITE and run_wsl(f"test -e {quote_wsl_path(graph_dir)}", check=False).returncode == 0:
        raise RuntimeError(f"Graph directory exists but has no valid manifest for {year}; set OVERWRITE=True to deliberately rebuild that year.")
    prepare_graph_dir(year)

    print(f"Build {year}: {PBF_BY_YEAR[year].name} -> {graph_dir}")
    if not RUN_BUILD:
        print("Dry run only. Set RUN_BUILD = True to start Docker.")
        continue

    write_status_row({
        "year": year,
        "status": "running",
        "started_at": started_at,
        "finished_at": "",
        "duration_minutes": "",
        "graph_path_wsl": graph_dir,
        "manifest_path_wsl": manifest_path,
        "error_message": "",
    })

    try:
        pbf_wsl_path = copy_pbf_to_wsl(year, staging_dir)
        start_valhalla_container(year, staging_dir)
        route_summaries = wait_until_valhalla_ready(year)
        finished_at = datetime.now().isoformat(timespec="seconds")
        write_wsl_manifest(year, staging_dir, pbf_wsl_path, route_summaries, started_at, finished_at)
        stop_valhalla_container(year)
        if OVERWRITE:
            remove_graph_dir(graph_dir)
        run_wsl(f"mv -- {quote_wsl_path(staging_dir)} {quote_wsl_path(graph_dir)}")

        duration_minutes = round((datetime.fromisoformat(finished_at) - datetime.fromisoformat(started_at)).total_seconds() / 60, 2)
        write_status_row({
            "year": year,
            "status": "done",
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_minutes": duration_minutes,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": "",
        })
        print(f"Done {year}: {route_summaries}")
    except Exception as error:
        stop_valhalla_container(year)
        finished_at = datetime.now().isoformat(timespec="seconds")
        duration_minutes = round((datetime.fromisoformat(finished_at) - datetime.fromisoformat(started_at)).total_seconds() / 60, 2)
        write_status_row({
            "year": year,
            "status": "failed",
            "started_at": started_at,
            "finished_at": finished_at,
            "duration_minutes": duration_minutes,
            "graph_path_wsl": graph_dir,
            "manifest_path_wsl": manifest_path,
            "error_message": repr(error),
        })
        raise

In [ ]:
read_status()